# HumorVibes — Panel Lab

**Kaggle's built-in LLM credits as the audience panel; local Gemma as the instrument.**

Theory (see THEORY.md in the mounted source): audiences are differently-tuned predictive meshes. Hosted models judge persona-conditioned; the small local Gemma *measures* — punchline surprisal S, and resolution R as the surprisal collapse a stated frame produces, net of a decoy-hint null control.

**Frame duel**: each hosted mesh writes its best one-line frame per joke; frames are scored by the measured collapse they cause. Explanation quality in nats, not vibes.

*Enable the free credits: Add-ons → Gemini (or attach a GEMINI_API_KEY secret). Without a key the notebook still runs the ground-truth-vs-self frame-gap experiment.*

In [ ]:
import glob, json, os, re, shutil, sys, time, urllib.request
apps = glob.glob('/kaggle/input/**/llm_panel.py', recursive=True)
assert apps, 'punchline-mesh-src dataset not attached'
SRC = os.path.dirname(apps[0])
sys.path.insert(0, SRC)
os.makedirs('/kaggle/working/research_out', exist_ok=True)
print('source:', SRC)

GEMINI_KEY = ''
try:
    from kaggle_secrets import UserSecretsClient
    usc = UserSecretsClient()
    for getter in (lambda: usc.get_gemini_api_key(),
                   lambda: usc.get_secret('GEMINI_API_KEY'),
                   lambda: usc.get_secret('GOOGLE_API_KEY')):
        try:
            GEMINI_KEY = getter() or ''
            if GEMINI_KEY: break
        except Exception: pass
except Exception as e:
    print('kaggle_secrets unavailable:', e)
if GEMINI_KEY:
    os.environ['GEMINI_API_KEY'] = GEMINI_KEY
    print('Gemini credits: AVAILABLE')
else:
    print('No Gemini key. Enable Add-ons -> Gemini (Kaggle built-in credits) and rerun for '
          'the hosted panel; continuing with the local-only frame-gap experiment.')

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
gcfg = [p for p in glob.glob('/kaggle/input/**/config.json', recursive=True) if 'gemma' in p.lower()]
MODEL_PATH = os.path.dirname(gcfg[0])
tok = AutoTokenizer.from_pretrained(MODEL_PATH)
def load_fallback(path):
    if torch.cuda.is_available():
        try:
            m = AutoModelForCausalLM.from_pretrained(path, torch_dtype=torch.float16, device_map='auto').eval()
            with torch.no_grad(): m(torch.tensor([[tok.bos_token_id or 2]]).to(m.device))
            return m
        except Exception as e:
            print('cuda failed ->cpu:', str(e)[:100]); torch.cuda.empty_cache()
    return AutoModelForCausalLM.from_pretrained(path, torch_dtype=torch.float32).eval()
model = load_fallback(MODEL_PATH)
print('instrument device:', model.device)

def nll_mean(context, continuation):
    ctx = tok(context, return_tensors='pt').input_ids
    cont = tok(continuation, add_special_tokens=False, return_tensors='pt').input_ids
    full = torch.cat([ctx, cont], dim=1).to(model.device)
    with torch.no_grad():
        lp = torch.log_softmax(model(full).logits.float(), dim=-1)
    n = ctx.shape[1]
    vals = [float(-lp[0, n+i-1, int(full[0, n+i])]) for i in range(cont.shape[1])]
    return sum(vals) / len(vals)

DECOY = 'It turns out this is really about quarterly regional cheese sales figures.'
def measure_frame(setup, punch, frame):
    S = nll_mean(setup + '\n', ' ' + punch)
    r_raw = max(0.0, S - nll_mean(setup + '\n(' + frame + ')\n', ' ' + punch))
    r_null = max(0.0, S - nll_mean(setup + '\n(' + DECOY + ')\n', ' ' + punch))
    return round(S, 3), round(max(0.0, r_raw - r_null), 3), round(r_raw, 3), round(r_null, 3)

In [ ]:
ITEMS = [
  ('speed_bumps', 'I told my therapist about my fear of speed bumps.', "She said I'm slowly getting over it.",
   "'Getting over it' is literal: the car physically drives over the speed bumps slowly."),
  ('lion_heart', 'My grandfather has the heart of a lion', 'and a lifetime ban from the zoo.',
   "He literally stole a lion's heart from the zoo, not the bravery metaphor."),
  ('ai_pm', 'I asked the AI project manager when the feature would ship.', 'It scheduled a meeting to align on what \'when\' means.',
   'The AI treats even the word when as a project requirement needing stakeholder alignment.'),
  ('nonsense_ctrl', 'I told my therapist about my fear of speed bumps.', 'The quarterly report shows strong regional cheese sales.',
   'NONE'),
]
def gen_local(prompt, max_new=60, temperature=0.3):
    ids = tok.apply_chat_template([{'role':'user','content':prompt}], return_tensors='pt', add_generation_prompt=True)
    if not torch.is_tensor(ids): ids = ids['input_ids']
    ids = ids.to(model.device)
    with torch.no_grad():
        out = model.generate(ids, max_new_tokens=max_new, do_sample=True, temperature=temperature,
                             top_p=0.95, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True).strip()

FRAME_ASK = ('A joke works because a hidden frame reinterprets the punchline - the fact that, once stated, '
             'makes the punchline the OBVIOUS next thing to say.\nJoke: {joke}\n'
             'Frame (ONE short sentence, no preamble; if none exists, write NONE):')

# Frame writers: local 2B always; Gemini models when credits exist
WRITERS = {'local-gemma-2-2b': lambda joke: gen_local(FRAME_ASK.format(joke=joke)).splitlines()[0].strip()}
if os.environ.get('GEMINI_API_KEY'):
    import llm_panel
    from llm_panel import PanelJudge, _dispatch
    for gm in ('gemini-2.5-flash', 'gemini-2.5-flash-lite'):
        judge = PanelJudge(f'gem-{gm}', 'openai-compat', gm,
                           'https://generativelanguage.googleapis.com/v1beta/openai', 'GEMINI_API_KEY')
        WRITERS[gm] = (lambda j: (lambda joke: (_dispatch(j, FRAME_ASK.format(joke=joke)) or '').strip().splitlines()[0].strip() if (_dispatch(j, FRAME_ASK.format(joke=joke)) or '').strip() else ''))(judge)
print('frame writers:', list(WRITERS))

## Frame duel: who explains the joke best, measured in nats

In [ ]:
duel = {}
for item_id, setup, punch, gt_frame in ITEMS:
    rows = {}
    if gt_frame != 'NONE':
        S, R, r_raw, r_null = measure_frame(setup, punch, gt_frame)
        rows['ground_truth'] = {'frame': gt_frame, 'S': S, 'R': R, 'R_raw': r_raw, 'R_null': r_null}
    for writer, fn in WRITERS.items():
        try:
            frame = fn(setup + ' ' + punch)
        except Exception as e:
            frame = ''
        if not frame or frame.upper().startswith('NONE'):
            rows[writer] = {'frame': frame or '(none)', 'S': None, 'R': 0.0}
            continue
        S, R, r_raw, r_null = measure_frame(setup, punch, frame)
        rows[writer] = {'frame': frame[:90], 'S': S, 'R': R, 'R_raw': r_raw, 'R_null': r_null}
    duel[item_id] = rows
    print('==', item_id)
    for w, r in sorted(rows.items(), key=lambda kv: -(kv[1].get('R') or 0)):
        print(f"   {w:22s} R={r.get('R')} :: {r['frame'][:70]}")
json.dump(duel, open('/kaggle/working/research_out/frame_duel.json', 'w'), indent=2)

## Persona panel (runs when credits exist)

In [ ]:
if os.environ.get('GEMINI_API_KEY'):
    os.environ['PANEL_GEMINI_MODELS'] = 'gemini-2.5-flash,gemini-2.5-flash-lite'
    import importlib, llm_panel
    importlib.reload(llm_panel)
    judges = [j for j in llm_panel.available_judges() if j.judge_id.startswith('gem-')]
    personas = ['NYC tech meetup crowd', 'retired farmers with no software exposure',
                'mixed-politics community-center audience']
    results = {}
    for item_id, setup, punch, _gt in ITEMS:
        votes = llm_panel.run_panel(setup + ' ' + punch, personas, judges=judges)
        results[item_id] = [v.__dict__ for v in votes]
        ok = [v for v in votes if v.ok]
        if ok:
            ov = [v.scores.get('overall', 0) for v in ok]
            print(f'{item_id:16s} {len(ok)}/{len(votes)} votes, overall mean {sum(ov)/len(ov):.2f}')
        else:
            print(f'{item_id:16s} all failed:', votes[0].error if votes else '?')
    json.dump(results, open('/kaggle/working/research_out/gemini_panel.json', 'w'), indent=2)
    print('wrote gemini_panel.json')
else:
    print('skipped (no Gemini credits attached — Add-ons -> Gemini, then rerun)')

## Reading the results
- **Ground-truth frames vs model frames**: the gap between `ground_truth` R and each writer's R is that writer's *explanation deficit* for the joke — understanding measured as the surprisal collapse its explanation produces in another mesh.
- **Nonsense control**: every honest writer should output NONE (R=0). A writer that invents a frame for nonsense — and especially one whose invented frame *measures* well — is confabulating; the decoy null keeps that in check.
- The persona panel extends the multi-mesh study (THEORY.md §7) using Kaggle's built-in LLM credits: validity (nonsense lowest), convergence by dimension, portability spread.